<a href="https://colab.research.google.com/github/mahieshwar-budati/Basic-Advance-RAG/blob/main/Hybrid_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "pymilvus[milvus_lite]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 17.8 MB/s eta 0:00:00


In [ ]:
from pymilvus import (
    MilvusClient,
    DataType,
    AnnSearchRequest,
    WeightedRanker
)
import random

# -----------------------
# Connect (Milvus Lite)
# -----------------------
client = MilvusClient(uri="./milvus_demo.db")

# Drop old collection
if client.has_collection("hybrid_demo"):
    client.drop_collection("hybrid_demo")

# -----------------------
# Create Schema
# -----------------------
schema = client.create_schema(auto_id=False)

schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("text_dense", DataType.FLOAT_VECTOR, dim=8)
schema.add_field("image_dense", DataType.FLOAT_VECTOR, dim=8)

# -----------------------
# Create Indexes
# -----------------------
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="text_dense",
    index_type="AUTOINDEX",
    metric_type="L2"
)

index_params.add_index(
    field_name="image_dense",
    index_type="AUTOINDEX",
    metric_type="L2"
)

client.create_collection(
    collection_name="hybrid_demo",
    schema=schema,
    index_params=index_params
)

# -----------------------
# Insert Sample Data
# -----------------------
def random_vector(dim):
    return [random.random() for _ in range(dim)]

data = [
    {"id": 0, "text_dense": random_vector(8), "image_dense": random_vector(8)},
    {"id": 1, "text_dense": random_vector(8), "image_dense": random_vector(8)},
    {"id": 2, "text_dense": random_vector(8), "image_dense": random_vector(8)},
]

client.insert("hybrid_demo", data)

client.flush("hybrid_demo")
client.load_collection("hybrid_demo")

print("Data inserted and collection loaded ✅")

# -----------------------
# Create Hybrid Requests
# -----------------------
query_text_vector = random_vector(8)
query_image_vector = random_vector(8)

# Text search
req1 = AnnSearchRequest(
    data=[query_text_vector],
    anns_field="text_dense",
    param={"nprobe": 10},
    limit=2
)

# Image search
req2 = AnnSearchRequest(
    data=[query_image_vector],
    anns_field="image_dense",
    param={"nprobe": 10},
    limit=2
)

requests = [req1, req2]

# -----------------------
# Weighted Ranker (required in Lite)
# -----------------------
ranker = WeightedRanker(0.5, 0.5)

# -----------------------
# Hybrid Search
# -----------------------
results = client.hybrid_search(
    collection_name="hybrid_demo",
    reqs=requests,
    ranker=ranker,
    limit=2
)

# -----------------------
# Print Results
# -----------------------
for hits in results:
    print("\nHybrid Search Results:")
    for hit in hits:
        print("ID:", hit.id)
        print("Distance:", hit.distance)
        print("-" * 20)

Data inserted and collection loaded ✅

Hybrid Search Results:
ID: 1
Distance: 0.596357524394989
--------------------
ID: 0
Distance: 0.35025882720947266
--------------------
